# PFE ML - Model Interrogation By SIREN

This Colab workbook loads the continuity-risk ML artifacts from Google Drive, ranks the trained runs, selects the best deployable model artifact, then scores one or more French company SIRENs.

Default storage root used by the training notebooks:

`/content/drive/MyDrive/PFE ML Data/pfe_data`

The prediction target is `continuity_risk_12m_label`: whether a company is likely to stop being active/open within the next 12 months from the selected feature cutoff year.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/PFE ML Data/pfe_data')
REPO_DIR = Path('/content/pfein/back_end')
REPO_URL = ''  # Optional: paste the GitHub repo URL here if /content/pfein is not already cloned.
INSTALL_REQUIREMENTS = True

cwd = Path.cwd()
if (cwd / 'collabs' / 'requirements-colab.txt').exists():
    REPO_DIR = cwd

if not REPO_DIR.exists():
    if REPO_URL:
        clone_target = REPO_DIR.parent
        clone_target.parent.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(['git', 'clone', REPO_URL, str(clone_target)])
    else:
        raise FileNotFoundError(
            'Repository not found at /content/pfein/back_end. Clone it first, or set REPO_URL in this cell.'
        )

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if INSTALL_REQUIREMENTS:
    requirements = REPO_DIR / 'collabs' / 'requirements-colab.txt'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])

print(f'Repo: {REPO_DIR}')
print(f'Drive root: {DRIVE_ROOT}')

## 1. Inspect Artifacts And Select The Deployable Model

The comparison table ranks all recorded runs. To keep the comparison fair, the default ranking is restricted to the same temporal split as the deployed metadata. If a run-specific `*.joblib` exists under `ml-artifacts/runs/<run_name>/`, the notebook can load that archived model. Otherwise it uses the root `ml-artifacts/model.joblib`, which is the deployable artifact produced by the latest training run.

In [ ]:
import json
import pandas as pd
from IPython.display import display

ARTIFACTS_DIR = DRIVE_ROOT / 'ml-artifacts'
DATA_LAKE = DRIVE_ROOT / 'data-lake'
MODEL_PATH = ARTIFACTS_DIR / 'model.joblib'
METADATA_PATH = ARTIFACTS_DIR / 'model_metadata.json'
COMPARISON_PATH = ARTIFACTS_DIR / 'model_run_comparison.csv'

SELECTION_METRIC = 'average_precision'
RESTRICT_TO_DEPLOYMENT_SPLIT = True

for required_path in [MODEL_PATH, METADATA_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required artifact: {required_path}')

metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
comparison = pd.read_csv(COMPARISON_PATH) if COMPARISON_PATH.exists() else pd.DataFrame()

def resolve_archived_model(run_name):
    if not isinstance(run_name, str) or not run_name:
        return None, None
    run_dir = ARTIFACTS_DIR / 'runs' / run_name
    if not run_dir.exists():
        return None, None
    preferred = [run_dir / 'model.joblib', run_dir / 'model.pkl']
    for candidate in preferred:
        if candidate.exists():
            return candidate, run_dir / 'metadata.json'
    matches = sorted(run_dir.glob('*.joblib')) + sorted(run_dir.glob('*.pkl'))
    if matches:
        return matches[0], run_dir / 'metadata.json'
    return None, None

SELECTED_MODEL_PATH = MODEL_PATH
SELECTED_METADATA_PATH = METADATA_PATH
selected_reason = 'Using root ml-artifacts/model.joblib because it is the deployable artifact produced by the latest training run.'

if not comparison.empty:
    metric = SELECTION_METRIC if SELECTION_METRIC in comparison.columns else 'roc_auc'
    pool = comparison.copy()
    if RESTRICT_TO_DEPLOYMENT_SPLIT and 'split_strategy' in pool.columns:
        pool = pool[pool['split_strategy'].eq(metadata.get('split_strategy'))].copy()
    if pool.empty:
        pool = comparison.copy()
    sort_columns = [c for c in [metric, 'roc_auc', 'trained_at'] if c in pool.columns]
    ascending = [False for _ in sort_columns]
    ranking = pool.sort_values(sort_columns, ascending=ascending) if sort_columns else pool
    best_row = ranking.iloc[0]
    for _, row in ranking.iterrows():
        candidate_model, candidate_metadata = resolve_archived_model(row.get('run_name'))
        if candidate_model is not None:
            SELECTED_MODEL_PATH = candidate_model
            SELECTED_METADATA_PATH = candidate_metadata if candidate_metadata and candidate_metadata.exists() else METADATA_PATH
            selected_reason = f'Using archived model for best ranked deployable run: {row.get("run_name")}'
            break
    display_columns = [
        'model_version', 'run_name', 'model_family', 'split_strategy',
        'average_precision', 'roc_auc', 'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5'
    ]
    display_columns = [c for c in display_columns if c in ranking.columns]
    print('Top runs by selected ranking policy:')
    display(ranking[display_columns].head(10))
    print('Best run in comparison table:', best_row.get('run_name'))
    print(f'Best {metric}:', best_row.get(metric))

selected_metadata = json.loads(SELECTED_METADATA_PATH.read_text(encoding='utf-8'))
print('\nSelected model artifact:', SELECTED_MODEL_PATH)
print('Selected metadata:', SELECTED_METADATA_PATH)
print('Selection reason:', selected_reason)
print('Model version:', selected_metadata.get('model_version'))
print('Run name:', selected_metadata.get('run_name'))
print('Family:', selected_metadata.get('model_family'))
print('Average precision:', selected_metadata.get('metrics', {}).get('average_precision'))
print('ROC AUC:', selected_metadata.get('metrics', {}).get('roc_auc'))

## 2. Load Model And Define SIREN Scoring Helpers

In [ ]:
import re
import duckdb
import joblib
import numpy as np
import pandas as pd

# Import custom sklearn pipeline classes so joblib can deserialize the fitted model.
from app.tools import train_continuity_model as _train_continuity_model  # noqa: F401

model = joblib.load(SELECTED_MODEL_PATH)

calibrator = None
calibrator_path = None
calibrator_candidates = [
    SELECTED_MODEL_PATH.parent / 'isotonic.joblib',
    SELECTED_MODEL_PATH.parent / 'isotonic_calibrator.joblib',
    SELECTED_MODEL_PATH.parent / 'platt.joblib',
    ARTIFACTS_DIR / 'isotonic.joblib',
    ARTIFACTS_DIR / 'isotonic_calibrator.joblib',
    ARTIFACTS_DIR / 'platt.joblib',
]
for candidate in calibrator_candidates:
    if candidate.exists():
        calibrator_path = candidate
        calibrator = joblib.load(candidate)
        break

if calibrator_path:
    print(f'Loaded probability calibrator: {calibrator_path}')
else:
    print('No probability calibrator found. Raw scores are useful for ranking; absolute probabilities may be overestimated.')

FEATURES_PATH = DATA_LAKE / 'features' / 'company_year_features'
FEATURES_GLOB = (FEATURES_PATH / '**' / '*.parquet').as_posix()
_FEATURES_GLOB_SQL = FEATURES_GLOB.replace(chr(39), chr(39) + chr(39))

if not FEATURES_PATH.exists():
    raise FileNotFoundError(f'Missing feature dataset: {FEATURES_PATH}')

def sql_literal(value):
    value = str(value).replace(chr(39), chr(39) + chr(39))
    return chr(39) + value + chr(39)

def normalize_siren(value):
    digits = re.sub(r'\D', '', str(value))
    if len(digits) != 9:
        raise ValueError(f'SIREN must contain exactly 9 digits, got {value!r}')
    return digits

def available_years_for_siren(siren):
    siren = normalize_siren(siren)
    query = f'''
        SELECT DISTINCT prediction_year
        FROM read_parquet('{_FEATURES_GLOB_SQL}', union_by_name=true)
        WHERE CAST(siren AS VARCHAR) = {sql_literal(siren)}
        ORDER BY prediction_year
    '''
    con = duckdb.connect()
    try:
        years = con.execute(query).df()['prediction_year'].dropna().astype(int).tolist()
    finally:
        con.close()
    return years

def load_company_feature_row(siren, prediction_year=None):
    siren = normalize_siren(siren)
    year_clause = '' if prediction_year is None else f'AND prediction_year = {int(prediction_year)}'
    query = f'''
        SELECT *
        FROM read_parquet('{_FEATURES_GLOB_SQL}', union_by_name=true)
        WHERE CAST(siren AS VARCHAR) = {sql_literal(siren)}
        {year_clause}
        ORDER BY prediction_year DESC
        LIMIT 1
    '''
    con = duckdb.connect()
    try:
        df = con.execute(query).df()
    finally:
        con.close()
    if df.empty:
        years = available_years_for_siren(siren)
        if years:
            raise ValueError(f'No feature row for SIREN {siren} and year {prediction_year}. Available years: {years}')
        raise ValueError(f'No feature row found for SIREN {siren}. Rebuild company_year_features if needed.')
    return df

def expected_feature_columns():
    columns = selected_metadata.get('feature_columns')
    if columns:
        return list(columns)
    if hasattr(model, 'feature_names_in_'):
        return list(model.feature_names_in_)
    if hasattr(model, 'named_steps'):
        for step in model.named_steps.values():
            if hasattr(step, 'feature_names_in_'):
                return list(step.feature_names_in_)
    raise RuntimeError('Could not determine model feature columns from metadata or fitted pipeline.')

FEATURE_COLUMNS = expected_feature_columns()
print(f'Model expects {len(FEATURE_COLUMNS)} feature columns.')

def align_features_for_model(df):
    aligned = df.copy()
    missing = [column for column in FEATURE_COLUMNS if column not in aligned.columns]
    for column in missing:
        aligned[column] = np.nan
    for column in aligned.columns:
        if pd.api.types.is_bool_dtype(aligned[column]):
            aligned[column] = aligned[column].astype(float)
    if missing:
        print('Missing model columns were added as NaN:', missing)
    return aligned[FEATURE_COLUMNS]

def maybe_calibrate(raw_probability):
    if calibrator is None:
        return None
    raw_probability = np.asarray(raw_probability, dtype=float)
    try:
        return calibrator.predict(raw_probability)
    except Exception:
        return calibrator.predict(raw_probability.reshape(-1, 1))

def risk_band(probability):
    if pd.isna(probability):
        return 'unknown'
    probability = float(probability)
    if probability >= 0.50:
        return 'high'
    if probability >= 0.30:
        return 'medium'
    if probability >= 0.10:
        return 'watch'
    return 'low'

def score_siren(siren, prediction_year=None):
    df = load_company_feature_row(siren, prediction_year)
    X = align_features_for_model(df)
    raw = model.predict_proba(X)[:, 1].astype(float)
    calibrated = maybe_calibrate(raw)
    out = pd.DataFrame(index=df.index)
    passthrough_columns = [
        'siren', 'prediction_year', 'prediction_date', 'company_name', 'activity_code',
        'legal_category_code', 'employee_size_bracket', 'administrative_status_at_cutoff',
        'company_age_years', 'legal_events_count_all', 'legal_risk_events_count_all',
        'legal_events_count_12m', 'radiation_events_count_all', 'days_since_last_legal_event',
        'annual_accounts_count_all', 'has_financial_data', 'latest_revenue', 'latest_net_result',
        'latest_debt_to_assets', 'years_since_last_financial_statement'
    ]
    for column in passthrough_columns:
        if column in df.columns:
            out[column] = df[column].values
    out['continuity_risk_12m_score_raw'] = raw
    if calibrated is not None:
        out['continuity_risk_12m_score_calibrated'] = np.asarray(calibrated, dtype=float)
        score_for_band = out['continuity_risk_12m_score_calibrated']
    else:
        score_for_band = out['continuity_risk_12m_score_raw']
    out['decision_threshold_0_5'] = out['continuity_risk_12m_score_raw'] >= 0.5
    out['risk_band'] = score_for_band.map(risk_band)
    out['selected_model_version'] = selected_metadata.get('model_version')
    out['selected_run_name'] = selected_metadata.get('run_name')
    train_end = selected_metadata.get('train_end_year')
    scored_year = int(out['prediction_year'].iloc[0]) if 'prediction_year' in out.columns else None
    if train_end is not None and scored_year is not None and scored_year > int(train_end):
        print(f'Warning: scoring prediction_year={scored_year}, which is after the training end year {train_end}.')
    return out.reset_index(drop=True)

def display_prediction_table(result):
    columns = [
        'siren', 'prediction_year', 'company_name', 'activity_code', 'legal_category_code',
        'administrative_status_at_cutoff', 'company_age_years',
        'continuity_risk_12m_score_raw', 'continuity_risk_12m_score_calibrated',
        'decision_threshold_0_5', 'risk_band'
    ]
    columns = [column for column in columns if column in result.columns]
    probability_columns = [column for column in columns if column.startswith('continuity_risk')]
    formatters = {column: '{:.2%}' for column in probability_columns}
    display(result[columns].style.format(formatters))

def predict_siren(siren, prediction_year=None):
    result = score_siren(siren, prediction_year)
    display_prediction_table(result)
    return result

def predict_many_sirens(sirens, prediction_year=None):
    frames = []
    errors = []
    for value in sirens:
        try:
            frames.append(score_siren(value, prediction_year))
        except Exception as exc:
            errors.append({'siren': str(value), 'error': str(exc)})
    result = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not result.empty:
        result = result.sort_values('continuity_risk_12m_score_raw', ascending=False).reset_index(drop=True)
        display_prediction_table(result)
    if errors:
        print('SIRENs not scored:')
        display(pd.DataFrame(errors))
    return result


## 3. Score One Company

Set `SIREN` to the 9-digit company identifier. Leave `PREDICTION_YEAR = None` to use the newest feature row available for that SIREN, or set a year such as `2024` for a fixed cutoff.

In [ ]:
SIREN = ''  # Example: '552120222'
PREDICTION_YEAR = None

if SIREN:
    prediction = predict_siren(SIREN, PREDICTION_YEAR)
else:
    print('Set SIREN to a 9-digit value, then run this cell.')

## 4. Optional Batch Scoring

In [ ]:
SIRENS = []  # Example: ['552120222', '542065305']
BATCH_PREDICTION_YEAR = None

if SIRENS:
    batch_predictions = predict_many_sirens(SIRENS, BATCH_PREDICTION_YEAR)
else:
    print('Add one or more SIRENs to SIRENS, then run this cell.')

## Interpretation Note

If no calibrator is present, the raw score should be treated mainly as a ranking signal. The training metadata shows that the model uses class balancing for a rare target, so raw probabilities can be higher than observed event rates. For user-facing probability text, add and load an isotonic or Platt calibrator from the calibration phase.